In [46]:
from IPython.display import display
from openai import OpenAI
from sqlalchemy import create_engine, inspect, text
import json
import os
import pandas as pd

In [ ]:
# ── Configuracion central ─────────────────────────────────────────
LM_STUDIO_HOST  = "http://192.168.1.100:1234"
LM_STUDIO_MODEL = "qwen/qwen3-5b"          # cambia al modelo cargado
# LM_STUDIO_MODEL = "lmstudio-community/mistral-7b-instruct-v0.3"

DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_NAME     = "app360"
DB_USER     = "app360"
DB_PASSWORD = "secure-db-password"
# ──────────────────────────────────────────────────────────────────


In [47]:
# Configuracion de LM Studio como servidor OpenAI-compatible
LM_STUDIO_HOST = os.getenv("LM_STUDIO_HOST", "http://192.168.1.100:1234")
LM_STUDIO_BASE_URL = f"{LM_STUDIO_HOST.rstrip('/')}/v1"
LM_STUDIO_MODEL = os.getenv("LM_STUDIO_MODEL", "qwen/qwen3.5-9b")
LM_STUDIO_API_KEY = os.getenv("LM_STUDIO_API_KEY", "lm-studio")

lm_client = OpenAI(
    base_url=LM_STUDIO_BASE_URL,
    api_key=LM_STUDIO_API_KEY,
    timeout=60.0,
 )

print(f"LM Studio base URL: {LM_STUDIO_BASE_URL}")
print(f"Modelo configurado: {LM_STUDIO_MODEL}")

LM Studio base URL: http://192.168.1.100:1234/v1
Modelo configurado: qwen/qwen3.5-9b


In [48]:
# Smoke test de conectividad con LM Studio
smoke_response = lm_client.chat.completions.create(
    model=LM_STUDIO_MODEL,
    messages=[
        {"role": "system", "content": "Eres un asistente breve y preciso."},
        {"role": "user", "content": "Responde solo con: LM Studio conectado."},
    ],
    temperature=0,
    max_tokens=20,
 )

print(smoke_response.choices[0].message.content)

In [49]:
# Credenciales por defecto para la instancia Docker app360
DB_USER = os.getenv("DB_USER", "app360")
DB_PASSWORD = os.getenv("DB_PASSWORD", "secure-db-password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "app360")

In [50]:
# Parametros de conexion para PostgreSQL Docker app360
USER = DB_USER
PASSWORD = DB_PASSWORD
HOST = DB_HOST
PORT = DB_PORT
DB = DB_NAME

if PASSWORD:
    db_url = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}"
else:
    db_url = f"postgresql+psycopg2://{USER}@{HOST}:{PORT}/{DB}"

engine = create_engine(db_url, pool_pre_ping=True)

# Prueba rapida de conexion
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database();")).scalar())

app360


In [51]:
prompt = str(input("Ingrese su pregunta: "))
prompt

'holaa'

In [ ]:
import re

user_question = str(globals().get("prompt") or "").strip()
if not user_question:
    user_question = str(input("Ingrese su pregunta: ")).strip()

if not user_question:
    raise ValueError("Debes ingresar una pregunta no vacia")

chat_response = lm_client.chat.completions.create(
    model=LM_STUDIO_MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Eres un analista de negocio. Responde en espanol claro y directo. "
                "No muestres razonamiento interno, thinking process, planes ni pasos intermedios. "
                "Entrega solo la respuesta final para el usuario."
            ),
        },
        {"role": "user", "content": user_question},
    ],
    temperature=0.2,
    max_tokens=520,
    extra_body={"enable_thinking": False},
)

msg = chat_response.choices[0].message
raw_content = msg.content or ""

# Qwen3 thinking mode: quitar bloques <think>...</think> del contenido
chat_answer = re.sub(r"<think>.*?</think>", "", raw_content, flags=re.DOTALL).strip()

# Si sigue vacio, intentar reasoning_content (algunos backends lo separan)
if not chat_answer:
    chat_answer = getattr(msg, "reasoning_content", None) or raw_content or "(sin respuesta)"

print(f"Usuario: {user_question}\n")
print(f"Asistente:\n{chat_answer}")


Usuario: holaa

Asistente:

